# 01 因子研究演示
历史回测不代表未来收益。

In [ ]:
import yaml, pandas as pd
from src.data.downloader import get_hs300_symbols, fetch_panel
from src.data.cleaner import clean_ohlcv, to_wide
from src.factors.momentum import momentum_20
from src.factors.reversal import reversal_5
from src.factors.volatility import volatility_20
from src.factors.volume_price import price_volume_divergence_20
from src.factors.turnover import turnover_20
from src.analysis.standardize import winsorized_zscore
from src.analysis.forward_returns import compute_forward_returns
from src.analysis.ic import compute_ic_series, summarize_ic
from src.analysis.quantile_returns import quantile_return_table
from src.factors.composite import equal_weight_composite
from src.backtest.engine import run_backtest
from src.backtest.metrics import calc_metrics
from src.analysis.report import generate_report

In [ ]:
cfg=yaml.safe_load(open('config.yaml','r',encoding='utf-8'))
symbols=get_hs300_symbols(cfg['data']['top_n'])
panel=fetch_panel(symbols, cfg['data']['start_date'], cfg['data']['end_date'])
panel={k: clean_ohlcv(v) for k,v in panel.items()}
close=to_wide(panel,'close'); volume=to_wide(panel,'volume')
turnover=to_wide(panel,'turnover') if 'turnover' in next(iter(panel.values())).columns else volume
returns=close.pct_change()

In [ ]:
factors={
'momentum_20':winsorized_zscore(momentum_20(close)),
'reversal_5':winsorized_zscore(reversal_5(close)),
'volatility_20':winsorized_zscore(volatility_20(returns)),
'price_volume_divergence_20':winsorized_zscore(price_volume_divergence_20(close, volume)),
'turnover_20':winsorized_zscore(turnover_20(turnover)),
}
score=equal_weight_composite(factors)
fwd=compute_forward_returns(close,[1,5,10])
ic=compute_ic_series(factors['momentum_20'], fwd['5D'])
print(summarize_ic(ic))
qt,ls=quantile_return_table(factors['momentum_20'], fwd['5D'])
bt=run_backtest(close, score, initial_capital=cfg['backtest']['initial_capital'], top_k=cfg['backtest']['top_k'])
print(calc_metrics(bt['strategy_return']))
generate_report(ic, qt.mean(), ls, out_dir='reports')